In [ ]:
from pathlib import Path

PROJECT_DIR = Path.cwd()
if PROJECT_DIR.name == "code":
    PROJECT_DIR = PROJECT_DIR.parent
RESULTS_DIR = PROJECT_DIR / "runs"
INPUT_DIR = PROJECT_DIR / "data"
if not INPUT_DIR.is_dir() or not RESULTS_DIR.is_dir():
    raise FileNotFoundError("Start Jupyter from the repository root or code directory")

import pandas as pd
import numpy as np

def read_results(method, num, scale):
    results = []
    for i in range(1, num + 1):
        with open(RESULTS_DIR / method / str(scale) / f"{i}.out", "r") as file:
            result = float(file.readline().strip())
            results.append(result)
    return results


def calculate_relative_difference(base_results, compare_results):
    return [((compare - base) / base) for base, compare in zip(base_results, compare_results)]

num = 20
scales = [5, 10, 20]

average_results = []
for scale in scales:
    dp_results = read_results("std", num, scale)
    sa_results = read_results("SA", num, scale)
    evol_results = read_results("evol", num, scale)
    reinforce_results = read_results("reinforce", num, scale)

    sa_relative_diff = calculate_relative_difference(dp_results, sa_results)
    evol_relative_diff = calculate_relative_difference(dp_results, evol_results)
    reinforce_relative_diff = calculate_relative_difference(dp_results, reinforce_results)

    average_sa = np.mean(sa_relative_diff) * 100
    average_evol = np.mean(evol_relative_diff) * 100
    average_reinforce = np.mean(reinforce_relative_diff) * 100

    average_results.append({
        "Scale": scale,
        "SA": f"{average_sa:.2f}%",
        "Evol": f"{average_evol:.2f}%",
        "Reinforce": f"{average_reinforce:.2f}%"
    })

    df = pd.DataFrame(average_results)
    df.set_index("Scale", inplace=True)

    # df.to_csv(f"performance_comparison_{scale}.csv")

scales_big = [50, 100]
for scale in scales_big:
    sa_results = read_results("SA", num, scale)
    evol_results = read_results("evol", num, scale)
    reinforce_results = read_results("reinforce", num, scale)

    sa_relative_diff = calculate_relative_difference(evol_results, sa_results)
    reinforce_relative_diff = calculate_relative_difference(evol_results, reinforce_results)

    average_sa = np.mean(sa_relative_diff) * 100
    average_evol = 0
    average_reinforce = np.mean(reinforce_relative_diff) * 100

    average_results.append({
        "Scale": scale,
        "SA": f"{average_sa:.2f}%",
        "Evol": f"{average_evol:.2f}%",
        "Reinforce": f"{average_reinforce:.2f}%"
    })

    df = pd.DataFrame(average_results)
    df.set_index("Scale", inplace=True)

    # df.to_csv(f"performance_comparison_{scale}.csv")

num = 4
scale = "spe"
std_results = read_results("std", num, scale)
sa_results = read_results("SA", num, scale)
evol_results = read_results("evol", num, scale)
reinforce_results = read_results("reinforce", num, scale)

sa_relative_diff = calculate_relative_difference(std_results, sa_results)
evol_relative_diff = calculate_relative_difference(std_results, evol_results)
reinforce_relative_diff = calculate_relative_difference(std_results, reinforce_results)

average_sa = np.mean(sa_relative_diff) * 100
average_evol = np.mean(evol_relative_diff) * 100
average_reinforce = np.mean(reinforce_relative_diff) * 100

average_results.append({
    "Scale": scale,
    "SA": f"{average_sa:.2f}%",
    "Evol": f"{average_evol:.2f}%",
    "Reinforce": f"{average_reinforce:.2f}%"
})

df = pd.DataFrame(average_results)
df.set_index("Scale", inplace=True)


print(df)
print("for Scale 50 and 100, the Evol algorithm is regarded as the baseline.")

In [ ]:
import matplotlib.pyplot as plt

def read_coordinates(num, scale):
    coordinates = []
    for i in range(1, num + 1):
        with open(INPUT_DIR / str(scale) / f"{i}.in", "r") as file:
            n = int(file.readline().strip())
            coords = []
            for _ in range(n):
                x, y = map(float, file.readline().strip().split())
                coords.append((x, y))
            coordinates.append(coords)
    return coordinates

def read_path(method, num, scale):
    paths = []
    for i in range(1, num + 1):
        with open(RESULTS_DIR / method / str(scale) / f"{i}.ans", "r") as file:
            path = list(map(int, file.readline().strip().split()))
            paths.append(path)
    return paths

def plot_path(ax, coords, path, label, title):
    x = [coords[i][0] for i in path]
    y = [coords[i][1] for i in path]
    ax.plot(x, y, 'o-', label=label)
    ax.set_xlabel('X')
    ax.set_ylabel('Y')
    ax.set_title(title)
    ax.legend()

num = 20
scales = [5, 10, 20]

#dp and SA
for scale in [20]:
    dp_results = read_path("std", num, scale)
    sa_results = read_path("SA", num, scale)
    coordinates = read_coordinates(num, scale)

    fig, axs = plt.subplots(2, 2, figsize=(15, 10))
    axs = axs.flatten()
    for i in range(4):
        coords = coordinates[i]
        dp_path = [_ - 1 for _ in dp_results[i]]
        sa_path = [_ - 1 for _ in sa_results[i]]

        plot_path(axs[i], coords, dp_path, "DP", f"DP compare SA {i+1}")
        plot_path(axs[i], coords, sa_path, "SA", f"DP compare SA {i+1}")

    plt.suptitle(f"Paths for Scale {scale}")
    plt.tight_layout(rect=[0, 0, 1, 0.96])
    plt.show()

#dp and reinforce
for scale in scales:
    dp_results = read_path("std", num, scale)
    reinforce_results = read_path("reinforce", num, scale)
    coordinates = read_coordinates(num, scale)

    fig, axs = plt.subplots(2, 2, figsize=(15, 10))
    axs = axs.flatten()
    for i in range(4):
        coords = coordinates[i]
        dp_path = [_ - 1 for _ in dp_results[i]]
        reinforce_path = [_ - 1 for _ in reinforce_results[i]]

        plot_path(axs[i], coords, dp_path, "DP", f"DP compare Reinforce {i+1}")
        plot_path(axs[i], coords, reinforce_path, "Reinforce", f"DP compare Reinforce {i+1}")

    plt.suptitle(f"Paths for Scale {scale}")
    plt.tight_layout(rect=[0, 0, 1, 0.96])
    plt.show()

#dp and evol
for scale in [20]:
    dp_results = read_path("std", num, scale)
    evol_results = read_path("evol", num, scale)
    coordinates = read_coordinates(num, scale)

    fig, axs = plt.subplots(2, 2, figsize=(15, 10))
    axs = axs.flatten()
    for i in range(4):
        coords = coordinates[i]
        dp_path = [_ - 1 for _ in dp_results[i]]
        evol_path = [_ - 1 for _ in evol_results[i]]

        plot_path(axs[i], coords, dp_path, "DP", f"DP compare Evol {i+1}")
        plot_path(axs[i], coords, evol_path, "Evol", f"DP compare Evol {i+1}")

    plt.suptitle(f"Paths for Scale {scale}")
    plt.tight_layout(rect=[0, 0, 1, 0.96])
    plt.show()

In [ ]:
num = 20

#SA, evol, and reinforce
for scale in scales_big:
    sa_results = read_path("SA", num, scale)
    evol_results = read_path("evol", num, scale)
    reinforce_results = read_path("reinforce", num, scale)
    coordinates = read_coordinates(num, scale)

    fig, axs = plt.subplots(2, 2, figsize=(15, 10))
    axs = axs.flatten()
    for i in range(4):
        coords = coordinates[i]
        sa_path = [_ - 1 for _ in sa_results[i]]
        evol_path = [_ - 1 for _ in evol_results[i]]
        reinforce_path = [_ - 1 for _ in reinforce_results[i]]

        plot_path(axs[i], coords, sa_path, "SA", f"Three methods comp {i+1}")
        plot_path(axs[i], coords, evol_path, "Evol", f"Three methods comp {i+1}")
        plot_path(axs[i], coords, reinforce_path, "Reinforce", f"Three methods comp {i+1}")

    plt.suptitle(f"Paths for Scale {scale}")
    plt.tight_layout(rect=[0, 0, 1, 0.96])
    plt.show()

In [ ]:
num = 4

std_results = read_path("std", num, "spe")

coordinates = read_coordinates(num, "spe")

#SA for special cases
sa_results = read_path("SA", num, "spe")

fig, axs = plt.subplots(2, 2, figsize=(25, 20))
axs = axs.flatten()
for i in range(4):
    coords = coordinates[i]

    std_path = [_ - 1 for _ in std_results[i]]
    std_path = {_: i for i, _ in enumerate(std_path)}
    std_path = sorted(std_path.items(), key=lambda x: x[0])
    std_path = [_[1] for _ in std_path]
    sa_path = [_ - 1 for _ in sa_results[i]]

    plot_path(axs[i], coords, std_path, "STD", f"SA {i+1}")
    plot_path(axs[i], coords, sa_path, "SA", f"SA {i+1}")

plt.suptitle(f"Paths for Scale 100")
plt.tight_layout(rect=[0, 0, 1, 0.96])
plt.show()

In [ ]:
#evol for special cases
evol_results = read_path("evol", num, "spe")

fig, axs = plt.subplots(2, 2, figsize=(25, 20))
axs = axs.flatten()
for i in range(4):
    coords = coordinates[i]

    std_path = [_ - 1 for _ in std_results[i]]
    std_path = {_: i for i, _ in enumerate(std_path)}
    std_path = sorted(std_path.items(), key=lambda x: x[0])
    std_path = [_[1] for _ in std_path]
    evol_path = [_ - 1 for _ in evol_results[i]]

    plot_path(axs[i], coords, std_path, "STD", f"Evol {i+1}")
    plot_path(axs[i], coords, evol_path, "Evol", f"Evol {i+1}")

plt.suptitle(f"Paths for Scale 100")
plt.tight_layout(rect=[0, 0, 1, 0.96])
plt.show()

In [ ]:
#reinforce for special cases
reinforce_results = read_path("reinforce", num, "spe")

fig, axs = plt.subplots(2, 2, figsize=(25, 20))
axs = axs.flatten()
for i in range(4):
    coords = coordinates[i]

    std_path = [_ - 1 for _ in std_results[i]]
    std_path = {_: i for i, _ in enumerate(std_path)}
    std_path = sorted(std_path.items(), key=lambda x: x[0])
    std_path = [_[1] for _ in std_path]
    reinforce_path = [_ - 1 for _ in reinforce_results[i]]

    plot_path(axs[i], coords, std_path, "STD", f"Reinforce {i+1}")
    plot_path(axs[i], coords, reinforce_path, "Reinforce", f"Reinforce {i+1}")

plt.suptitle(f"Paths for Scale 100")
plt.tight_layout(rect=[0, 0, 1, 0.96])
plt.show()
